In [1]:
# Install and import necessary packages
!pip install nltk pandas -q
import nltk
import pandas as pd
import re

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

print("Libraries and dependencies loaded successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Libraries and dependencies loaded successfully!


In [2]:
# Create a sample product feedback dataframe
data = {
    'Review_ID': ['R001', 'R002', 'R003', 'R004', 'R005', 'R006'],
    'Source': ['E-commerce', 'Survey', 'Support', 'E-commerce', 'Survey', 'Support'],
    'Rating': [5, 2, 4, 1, 5, 3],
    'Feedback_Text': [
        "The build quality of the blender is fantastic, but the power cord is way too short.",
        "Extremely overpriced for the flimsy plastic packaging it arrived in. Returned immediately.",
        "Easy to set up and use right out of the box. Great value for money.",
        "The box was completely crushed and the item arrived broken. Terrible experience.",
        "Super sturdy material and works like a charm. Highly recommend!",
        "Manual instructions were very confusing, took hours to figure out the setup."
    ]
}

df = pd.DataFrame(data)
df.head()

,Review_ID,Source,Rating,Feedback_Text
0,R001,E-commerce,5,"The build quality of the blender is fantastic,..."
1,R002,Survey,2,Extremely overpriced for the flimsy plastic pa...
2,R003,Support,4,Easy to set up and use right out of the box. G...
3,R004,E-commerce,1,The box was completely crushed and the item ar...
4,R005,Survey,5,Super sturdy material and works like a charm. ...


In [3]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase and remove HTML/URLs
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenize, remove stopwords, and lemmatize
    tokens = text.split()
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(cleaned_tokens)

# Apply preprocessing to the dataframe
df['Cleaned_Text'] = df['Feedback_Text'].apply(preprocess_text)
df[['Review_ID', 'Feedback_Text', 'Cleaned_Text']]

,Review_ID,Feedback_Text,Cleaned_Text
0,R001,"The build quality of the blender is fantastic,...",build quality blender fantastic power cord way...
1,R002,Extremely overpriced for the flimsy plastic pa...,extremely overpriced flimsy plastic packaging ...
2,R003,Easy to set up and use right out of the box. G...,easy set use right box great value money
3,R004,The box was completely crushed and the item ar...,box completely crushed item arrived broken ter...
4,R005,Super sturdy material and works like a charm. ...,super sturdy material work like charm highly r...
5,R006,"Manual instructions were very confusing, took ...",manual instruction confusing took hour figure ...


In [4]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Apply sentiment analysis
df['Sentiment'] = df['Feedback_Text'].apply(get_sentiment)
df[['Review_ID', 'Rating', 'Sentiment', 'Feedback_Text']]

,Review_ID,Rating,Sentiment,Feedback_Text
0,R001,5,Positive,"The build quality of the blender is fantastic,..."
1,R002,2,Neutral,Extremely overpriced for the flimsy plastic pa...
2,R003,4,Positive,Easy to set up and use right out of the box. G...
3,R004,1,Negative,The box was completely crushed and the item ar...
4,R005,5,Positive,Super sturdy material and works like a charm. ...
5,R006,3,Negative,"Manual instructions were very confusing, took ..."


In [5]:
# Define category keywords for aspect extraction
aspect_keywords = {
    'Quality': ['quality', 'build', 'material', 'sturdy', 'broken', 'durable', 'plastic'],
    'Pricing': ['price', 'priced', 'value', 'money', 'expensive', 'cheap', 'cost', 'overpriced'],
    'Packaging': ['packaging', 'box', 'arrived', 'crushed', 'wrapped'],
    'Usability': ['setup', 'use', 'manual', 'instructions', 'easy', 'confusing', 'cord']
}

def extract_aspects(text):
    matched_aspects = []
    text_lower = text.lower()
    for aspect, keywords in aspect_keywords.items():
        if any(kw in text_lower for kw in keywords):
            matched_aspects.append(aspect)
    return matched_aspects if matched_aspects else ['General']

# Apply aspect mapping
df['Aspects'] = df['Feedback_Text'].apply(extract_aspects)
df[['Review_ID', 'Aspects', 'Feedback_Text']]

,Review_ID,Aspects,Feedback_Text
0,R001,"[Quality, Usability]","The build quality of the blender is fantastic,..."
1,R002,"[Quality, Pricing, Packaging]",Extremely overpriced for the flimsy plastic pa...
2,R003,"[Pricing, Packaging, Usability]",Easy to set up and use right out of the box. G...
3,R004,"[Quality, Packaging]",The box was completely crushed and the item ar...
4,R005,[Quality],Super sturdy material and works like a charm. ...
5,R006,[Usability],"Manual instructions were very confusing, took ..."


In [6]:
# Summary statistics
print("=== Sentiment Distribution ===")
print(df['Sentiment'].value_counts(), "\n")

print("=== Overall Metrics ===")
print(f"Total Reviews: {len(df)}")
print(f"Average Rating: {df['Rating'].mean():.2f} / 5.0")

# View final consolidated dataframe
df[['Review_ID', 'Rating', 'Sentiment', 'Aspects', 'Feedback_Text']]

=== Sentiment Distribution ===
Sentiment
Positive    3
Negative    2
Neutral     1
Name: count, dtype: int64 

=== Overall Metrics ===
Total Reviews: 6
Average Rating: 3.33 / 5.0


,Review_ID,Rating,Sentiment,Aspects,Feedback_Text
0,R001,5,Positive,"[Quality, Usability]","The build quality of the blender is fantastic,..."
1,R002,2,Neutral,"[Quality, Pricing, Packaging]",Extremely overpriced for the flimsy plastic pa...
2,R003,4,Positive,"[Pricing, Packaging, Usability]",Easy to set up and use right out of the box. G...
3,R004,1,Negative,"[Quality, Packaging]",The box was completely crushed and the item ar...
4,R005,5,Positive,[Quality],Super sturdy material and works like a charm. ...
5,R006,3,Negative,[Usability],"Manual instructions were very confusing, took ..."
